# 🎯 Technique 81: Token Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/81_token_optimization.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  **Technique #:** 81  **Difficulty:** Intermediate

## 📋 Description

Token Optimization focuses on understanding and minimizing token usage in prompts and responses. Since LLM APIs charge by the token, optimizing token usage directly impacts cost. This technique involves understanding tokenization, identifying token inefficiencies, and applying strategies to reduce token count while maintaining quality.

**When to use:**
- Cost optimization for high-volume applications
- Staying within model context limits
- Improving latency (fewer tokens = faster processing)
- Budget-conscious prototyping and development
- Edge/mobile deployments with limited resources

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                    TOKENIZATION BASICS                       │
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  TEXT  →  [Tokenizer]  →  TOKENS  →  [Model]  →  OUTPUT     │
│  "Hello"      ↓         [15496]                              │
│  " world"     ↓         [995]                                │
│               ↓                                              │
│  Common words = 1 token    Rare words = 2-4 tokens           │
│  "the" = 1 token          "tokenization" = 3-4 tokens       │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│              TOKEN OPTIMIZATION STRATEGIES                   │
└─────────────────────────────────────────────────────────────┘
                            │
        ┌───────────────────┼───────────────────┐
        ▼                   ▼                   ▼
┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│   VOCABULARY │   │   STRUCTURE  │   │   RESPONSE   │
│   CHOICES    │   │   EFFICIENCY │   │   CONTROL    │
└──────────────┘   └──────────────┘   └──────────────┘
        │                   │                   │
        ▼                   ▼                   ▼
   - Common words       - Lists vs prose    - max_tokens
   - Avoid rare terms   - Abbreviations     - stop sequences
   - Consistent style   - Active voice      - Response format
   - Standard spelling  - Remove redundancy - Structured output

COST FORMULA:
Total Cost = (Input Tokens + Output Tokens) × Price per Token
           = (Prompt + Context + Response) × Rate
```

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai tiktoken

import openai
import tiktoken
from typing import List, Dict, Tuple
from dataclasses import dataclass
from getpass import getpass
import json

In [ ]:
# Configure API
openai.api_key = getpass("Enter your OpenAI API key: ")

# Initialize tokenizer
encoding = tiktoken.encoding_for_model("gpt-4")

## 🛠️ Implementation: Token Analyzer & Optimizer

In [ ]:
@dataclass
class TokenAnalysis:
    """Detailed token analysis result."""
    text: str
    token_count: int
    char_count: int
    tokens_per_char: float
    estimated_cost_input: float  # GPT-4 rate
    estimated_cost_output: float
    token_breakdown: List[Tuple[str, int]]


class TokenOptimizer:
    """Tools for analyzing and optimizing token usage."""
    
    # Pricing per 1K tokens (as of 2024)
    PRICING = {
        "gpt-4": {"input": 0.03, "output": 0.06},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-3.5-turbo": {"input": 0.0015, "output": 0.002},
    }
    
    def __init__(self, model: str = "gpt-4"):
        self.model = model
        self.encoding = tiktoken.encoding_for_model(model)
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in text."""
        return len(self.encoding.encode(text))
    
    def get_token_breakdown(self, text: str) -> List[Tuple[str, int]]:
        """Get word-level token breakdown."""
        tokens = self.encoding.encode(text)
        decoded = [self.encoding.decode([t]) for t in tokens]
        
        # Group by words
        word_tokens = {}
        current_word = ""
        current_count = 0
        
        for token in decoded:
            if token.strip() and not token.startswith(' '):
                if current_word:
                    word_tokens[current_word] = word_tokens.get(current_word, 0) + current_count
                current_word = token
                current_count = 1
            else:
                current_count += 1
        
        if current_word:
            word_tokens[current_word] = word_tokens.get(current_word, 0) + current_count
        
        return sorted(word_tokens.items(), key=lambda x: x[1], reverse=True)
    
    def analyze(self, text: str, model: str = None) -> TokenAnalysis:
        """Perform comprehensive token analysis."""
        model = model or self.model
        token_count = self.count_tokens(text)
        char_count = len(text)
        
        pricing = self.PRICING.get(model, self.PRICING["gpt-4"])
        
        return TokenAnalysis(
            text=text,
            token_count=token_count,
            char_count=char_count,
            tokens_per_char=token_count / char_count if char_count > 0 else 0,
            estimated_cost_input=(token_count / 1000) * pricing["input"],
            estimated_cost_output=(token_count / 1000) * pricing["output"],
            token_breakdown=self.get_token_breakdown(text)[:10]
        )
    
    def compare_variants(self, variants: List[str], labels: List[str] = None) -> Dict:
        """Compare multiple text variants."""
        results = []
        for i, text in enumerate(variants):
            analysis = self.analyze(text)
            results.append({
                "label": labels[i] if labels else f"Variant {i+1}",
                "tokens": analysis.token_count,
                "chars": analysis.char_count,
                "ratio": round(analysis.tokens_per_char, 3),
                "cost_input": f"${analysis.estimated_cost_input:.4f}",
                "cost_output": f"${analysis.estimated_cost_output:.4f}"
            })
        return results
    
    def estimate_monthly_cost(
        self, 
        daily_requests: int, 
        avg_input_tokens: int, 
        avg_output_tokens: int,
        model: str = None
    ) -> Dict:
        """Estimate monthly API costs."""
        model = model or self.model
        pricing = self.PRICING.get(model, self.PRICING["gpt-4"])
        
        daily_input = daily_requests * avg_input_tokens
        daily_output = daily_requests * avg_output_tokens
        
        monthly_input = daily_input * 30
        monthly_output = daily_output * 30
        
        cost_input = (monthly_input / 1000) * pricing["input"]
        cost_output = (monthly_output / 1000) * pricing["output"]
        
        return {
            "daily_requests": daily_requests,
            "monthly_input_tokens": monthly_input,
            "monthly_output_tokens": monthly_output,
            "input_cost": cost_input,
            "output_cost": cost_output,
            "total_cost": cost_input + cost_output
        }

## 💡 Basic Example: Token Analysis

In [ ]:
# Initialize optimizer
optimizer = TokenOptimizer(model="gpt-4")

# Example texts to analyze
texts = {
    "Common words": "The quick brown fox jumps over the lazy dog.",
    "Rare words": "The expeditious auburn vulpine leaps above the indolent canine.",
    "Numbers": "The price is $1,234.56 for 100 items.",
    "Code": "def hello_world():\n    print('Hello, World!')",
    "Unicode": "Café résumé naïve",
}

print("🔍 Token Analysis Examples")
print("="*60)

for name, text in texts.items():
    analysis = optimizer.analyze(text)
    print(f"\n{name}:")
    print(f"  Text: '{text}'")
    print(f"  Characters: {analysis.char_count}")
    print(f"  Tokens: {analysis.token_count}")
    print(f"  Tokens/Char: {analysis.tokens_per_char:.3f}")
    print(f"  Cost (input): {analysis.estimated_cost_input:.6f}")

In [ ]:
# Compare prompt variants
print("\n" + "="*60)
print("📊 PROMPT VARIANT COMPARISON")
print("="*60)

variants = [
    """Please analyze the following text and provide a detailed summary of the main points, 
including key themes, important details, and overall sentiment. Make sure to be thorough 
and comprehensive in your analysis.""",
    """Summarize this text with key themes, details, and sentiment.""",
    """Summarize: key themes, details, sentiment.""",
]

labels = ["Verbose", "Concise", "Minimal"]

comparison = optimizer.compare_variants(variants, labels)

for result in comparison:
    print(f"\n{result['label']}:")
    print(f"  Tokens: {result['tokens']}")
    print(f"  Input Cost: {result['cost_input']}")
    print(f"  Text: '{variants[labels.index(result['label'])][:50]}...'")

# Calculate savings
verbose_tokens = comparison[0]['tokens']
minimal_tokens = comparison[2]['tokens']
savings = ((verbose_tokens - minimal_tokens) / verbose_tokens) * 100
print(f"\n💰 Savings: {savings:.1f}% fewer tokens with minimal version")

## 🌍 Real-World Example: Cost Estimation & Optimization

In [ ]:
# Real-world: Estimating costs for a production application

print("💵 PRODUCTION COST ESTIMATION")
print("="*60)

# Scenario: Customer support chatbot
scenarios = [
    {
        "name": "Startup (Low Volume)",
        "daily_requests": 1000,
        "avg_input_tokens": 500,
        "avg_output_tokens": 300
    },
    {
        "name": "SMB (Medium Volume)",
        "daily_requests": 10000,
        "avg_input_tokens": 800,
        "avg_output_tokens": 400
    },
    {
        "name": "Enterprise (High Volume)",
        "daily_requests": 100000,
        "avg_input_tokens": 1000,
        "avg_output_tokens": 500
    }
]

models = ["gpt-4", "gpt-4o", "gpt-4o-mini"]

for scenario in scenarios:
    print(f"\n{'='*60}")
    print(f"📊 {scenario['name']}")
    print(f"{'='*60}")
    print(f"Daily requests: {scenario['daily_requests']:,}")
    print(f"Avg input/output: {scenario['avg_input_tokens']}/{scenario['avg_output_tokens']} tokens")
    
    print("\nMonthly Cost by Model:")
    for model in models:
        opt = TokenOptimizer(model=model)
        cost = opt.estimate_monthly_cost(
            scenario['daily_requests'],
            scenario['avg_input_tokens'],
            scenario['avg_output_tokens'],
            model
        )
        print(f"  {model:15}: ${cost['total_cost']:,.2f}")

In [ ]:
# Optimization impact analysis
print("\n" + "="*60)
print("📉 OPTIMIZATION IMPACT ANALYSIS")
print("="*60)

# Before and after optimization
before = {
    "avg_input_tokens": 1000,
    "avg_output_tokens": 500,
    "daily_requests": 50000
}

after = {
    "avg_input_tokens": 600,  # 40% reduction through prompt optimization
    "avg_output_tokens": 400,  # 20% reduction through response constraints
    "daily_requests": 50000
}

gpt4_optimizer = TokenOptimizer("gpt-4")

cost_before = gpt4_optimizer.estimate_monthly_cost(
    before['daily_requests'],
    before['avg_input_tokens'],
    before['avg_output_tokens']
)

cost_after = gpt4_optimizer.estimate_monthly_cost(
    after['daily_requests'],
    after['avg_input_tokens'],
    after['avg_output_tokens']
)

savings = cost_before['total_cost'] - cost_after['total_cost']
savings_percent = (savings / cost_before['total_cost']) * 100

print(f"\nBefore Optimization:")
print(f"  Monthly cost: ${cost_before['total_cost']:,.2f}")
print(f"  Input tokens/month: {cost_before['monthly_input_tokens']:,}")

print(f"\nAfter Optimization:")
print(f"  Monthly cost: ${cost_after['total_cost']:,.2f}")
print(f"  Input tokens/month: {cost_after['monthly_input_tokens']:,}")

print(f"\n💰 SAVINGS:")
print(f"  Monthly: ${savings:,.2f}")
print(f"  Percentage: {savings_percent:.1f}%")
print(f"  Annual: ${savings * 12:,.2f}")

## ⚠️ Failure Case: Over-Optimization

In [ ]:
print("⚠️ FAILURE CASE: Token Over-Optimization\n")
print("="*60)

# Example of over-optimized prompt that loses meaning
proper_prompt = """
Analyze the sentiment of the following customer review and classify it as 
POSITIVE, NEGATIVE, or NEUTRAL. Provide a brief explanation for your classification.

Review: The product quality exceeded my expectations.
"""

over_optimized = """Sent:POS/NEG/NEU.Exp brief.Rev:Prod qual expec."""

print("Proper Prompt:")
print(proper_prompt)
print(f"Tokens: {optimizer.count_tokens(proper_prompt)}\n")

print("Over-Optimized Prompt:")
print(over_optimized)
print(f"Tokens: {optimizer.count_tokens(over_optimized)}\n")

print("="*60)
print("ANALYSIS OF FAILURE:")
print("="*60)
print("""
PROBLEMS WITH OVER-OPTIMIZATION:

1. AMBIGUITY
   - "Exp brief" could mean "expect brief" or "explain briefly"
   - Abbreviations confuse the model

2. CONTEXT LOSS
   - "Prod qual expec" loses semantic meaning
   - Model may not understand the task

3. INCONSISTENT OUTPUT
   - Unpredictable response format
   - Difficult to parse results

4. MAINTAINABILITY
   - Impossible for humans to understand
   - Difficult to modify or debug

BETTER APPROACH:
- Optimize to 60-70% of original, not 20%
- Keep key instructions explicit
- Maintain human readability
- Test thoroughly after optimization
""")

## 📊 Token Efficiency Benchmarks

| Model | Input Cost/1K | Output Cost/1K | Context Window |
|-------|---------------|----------------|----------------|
| GPT-4 | $0.03 | $0.06 | 8K/32K |
| GPT-4o | $0.005 | $0.015 | 128K |
| GPT-4o-mini | $0.00015 | $0.0006 | 128K |
| GPT-3.5-turbo | $0.0015 | $0.002 | 16K |

**Optimization Targets:**
- Aim for 30-50% token reduction in prompts
- Use response constraints (max_tokens, format)
- Cache repeated context when possible
- Consider cheaper models for simple tasks

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
║              🎮 TOKEN OPTIMIZATION PLAYGROUND                     ║
╚══════════════════════════════════════════════════════════════════╝

# Analyze your own text:
YOUR_TEXT = """
[Paste your prompt or text here to analyze]
"""

YOUR_MODEL = "gpt-4o"  # Change to analyze different models

# Run analysis (uncomment to execute):
# my_optimizer = TokenOptimizer(YOUR_MODEL)
# analysis = my_optimizer.analyze(YOUR_TEXT)
# print(f"Tokens: {analysis.token_count}")
# print(f"Cost (input): ${analysis.estimated_cost_input:.6f}")
# print(f"Cost (output): ${analysis.estimated_cost_output:.6f}")
# print("\nTop token consumers:")
# for word, count in analysis.token_breakdown:
#     print(f"  {word}: {count} tokens")

## 💡 Tips & Tricks

### Token-Efficient Writing

| Instead of... | Use... | Token Savings |
|---------------|--------|---------------|
| "in order to" | "to" | ~2 tokens |
| "due to the fact that" | "because" | ~4 tokens |
| "at this point in time" | "now" | ~4 tokens |
| "is able to" | "can" | ~2 tokens |
| "in the event that" | "if" | ~3 tokens |

### Model Selection for Cost
- Use GPT-4o-mini for simple classification tasks
- Use GPT-4o for most production workloads
- Reserve GPT-4 for complex reasoning only

### Response Optimization
- Set `max_tokens` based on expected response length
- Use `stop` sequences to prevent runaway generation
- Request structured formats (JSON, bullet points)
- Specify desired response length explicitly

## 📚 References

1. [OpenAI Tokenizer](https://platform.openai.com/tokenizer) - Visualize tokenization
2. [Tiktoken Documentation](https://github.com/openai/tiktoken) - Fast BPE tokenizer
3. [OpenAI Pricing](https://openai.com/pricing) - Current token pricing
4. [Tokenization in LLMs](https://arxiv.org/abs/2306.02989) - Research paper
5. [Neural Machine Translation with Byte-Level Subwords](https://arxiv.org/abs/1909.03341) - BPE paper